# K2460 run viewer

This notebook is **read-only**: it only reads saved HDF5 files and never connects
to the instrument or RE Manager, writes data, or repairs files. It is safe to run
while the QueueServer writes. Set `DATA_DIR` to the session's data directory on
**local disk** (required for HDF5 SWMR).

Use the repository environment with the `experiment` extra. Times and values come
from saved records. Charge is the signed trapezoidal integral of measured current
from the first record (no estimate before it); negative current decreases charge.
Plots use bin extrema to keep short pulses visible when a run has more points
than they draw; the full view is exact up to `FULL_VIEW_POINTS` per trace.

An `incomplete` outcome can mean a live run or a crashed writer. The viewer cannot
distinguish them. Stopping the Follow cell only stops watching.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import display

from ophyd_electrochemistry.sequences.run_file import read_run_file
from ophyd_electrochemistry.sequences.viewer import (
    follow,
    latest_run,
    overlay_figure,
    run_figure,
    run_index,
)

pio.renderers.default = "plotly_mimetype"
DATA_DIR = Path.home() / "k2460-data"
FOLLOW = None  # measurement file name, or None for the newest run
COMPARE = []  # file names to overlay
X_AXIS = "charge"  # or "time"
FULL_VIEW_POINTS = 200_000  # per trace; exact below this, extrema-thinned above

## Overview

One row per HDF5 file, newest modified first. Rerun this cell to refresh the list.
`start_utc` uses the acquisition UTC anchor when available, otherwise file start.
Unreadable files stay in the list with an explanation.

In [ ]:
index = run_index(DATA_DIR)
display(index)

## Follow a run

Uses `FOLLOW`, or the newest file when it is `None`. This cell keeps running until
the run ends and replaces one plot in place every two seconds. Until the run
has readings (for example while it waits for an external START) it shows a
one-line status instead; the plot appears with the first readings. Use the notebook's
**stop button** to leave it early; an incomplete file from a crashed writer also
needs the stop button. Set `FOLLOW` to a file name to select another run.

In [ ]:
selected_path = DATA_DIR / FOLLOW if FOLLOW is not None else latest_run(DATA_DIR)
if selected_path is None:
    print("No HDF5 runs found. Check DATA_DIR and refresh the overview.")
else:
    outcome = follow(selected_path)
    print(f"Last observed outcome: {outcome}")

## One run at full view

Review the selected run at full resolution, with its events table. Rerun after
stopping Follow to read the latest snapshot. `X_AXIS` selects time or signed
charge; Follow always uses time. Runs up to `FULL_VIEW_POINTS` records per trace are drawn sample by sample; longer
runs are thinned by bin extrema (pulses stay visible), because a browser plot of
several hundred thousand points stalls Jupyter. Zoom shows the same data.

In [ ]:
if selected_path is not None:
    saved = read_run_file(selected_path)
    full_figure = run_figure(selected_path, x=X_AXIS, max_points=FULL_VIEW_POINTS)
    full_figure.show()
    display(pd.DataFrame(saved["events"]))
else:
    print("Select a run in Follow first.")

## Compare runs

Overlay voltage for `COMPARE`, or the two newest finished runs (including stopped,
aborted or errored runs). Each run starts at zero on the chosen axis. Legends show
measurement and sample IDs; inspect the overview for outcomes before comparing.
Only the display is decimated: charge always uses all saved samples.

In [ ]:
finished = run_index(DATA_DIR)
compare_paths = (
    [DATA_DIR / name for name in COMPARE]
    if COMPARE
    else finished.loc[
        finished["outcome"].isin(["completed", "stopped_cutoff", "aborted", "error"]), "file"
    ]
    .head(2)
    .tolist()
)
if compare_paths:
    comparison = overlay_figure(compare_paths, x=X_AXIS)
    comparison.show()
else:
    print("No finished runs to compare. Set COMPARE to file names to choose runs.")